# NGC 1068 low-flux Case 1 versus Testagrossa-reference CPL

Explicit NGC 4151-style comparison without the compact NGC 1068 helper.


In [ ]:
from pathlib import Path
import json

import sys
UTILS_DIR = Path(r"/Users/parshadkp/Software/cosipy/docs/tutorials/spectral_fits/continuum_fit/AGN/Fluctuate_True")
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import astropy.units as u
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from astromodels import Cutoff_powerlaw, Model, PointSource
from threeML import DataList, JointLikelihood
from cosipy.event_selection import GoodTimeInterval
from agn_cosi_fit_utils import (
    COSIPlugin, cosi_source_detection_ts, load_agn_manifest_histograms,
    make_cosi_background_parameter, open_spacecraft_history,
    save_agn_fit_summary, scale_spacecraft_livetime,
)
from agn_sed_ensemble import (
    classify_representative_sed, ensemble_summary_to_sed_dataframe,
    fit_representative_sed, run_or_load_manifest_sed_ensemble,
)
SED_KEV_TO_ERG=u.keV.to(u.erg); KEV_TO_MEV=u.keV.to(u.MeV)
%matplotlib inline


# Model and input files


In [ ]:
BACKGROUND_PSEUDOCOUNT=1e-12
SED_ENSEMBLE_RECOMPUTE=False
N_SED_BINS=10
manifest_paths={
    "low_flux_3m": Path(r"/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/GammaRay/Paper_Models/Fluctuate_True/Sensitivity_Ensemble/NGC1068_Case1_lowFlux100/NGC1068_Case1_lowFlux100_CPL_median_realization_3months.json"),
    "low_flux_24m": Path(r"/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/GammaRay/Paper_Models/Fluctuate_True/Sensitivity_Ensemble/NGC1068_Case1_lowFlux100/NGC1068_Case1_lowFlux100_CPL_median_realization_24months.json"),
    "testagrossa_3m": Path(r"/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/GammaRay/Paper_Models/Fluctuate_True/Sensitivity_Ensemble/NGC1068_TestagrossaReference/NGC1068_TestagrossaReference_CPL_median_realization_3months.json"),
    "testagrossa_24m": Path(r"/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Radio_Quiet_AGN/GammaRay/Paper_Models/Fluctuate_True/Sensitivity_Ensemble/NGC1068_TestagrossaReference/NGC1068_TestagrossaReference_CPL_median_realization_24months.json"),
}
PLOT_DIR=Path(r"/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/Papers/AGN_Corona_EC_Overleaf/Plots/Fluctuate_True")
FIT_SUMMARY_PATH=Path(r"/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/Papers/AGN_Corona_EC_Overleaf/Fits/Fluctuate_True/NGC1068_Case1_lowFlux100_vs_Testagrossa_fit_summary_3_24Months.txt")
PLOT_DIR.mkdir(parents=True,exist_ok=True); FIT_SUMMARY_PATH.parent.mkdir(parents=True,exist_ok=True)


# Spectral fitting


In [ ]:
def fit_cpl_manifest(manifest_path, dataset_name):
    with Path(manifest_path).open() as stream:
        manifest=json.load(stream)
    case=manifest.get("resolved_case_at_threshold",manifest.get("resolved_case"))
    exposure=int(manifest["exposure_months"])
    manifest,source_expectation,data_hist,background=load_agn_manifest_histograms(
        manifest_path,manifest["background_file_3m"],exposure,
    )
    spec=case["primary"]
    injected=Cutoff_powerlaw()
    injected.K.value=float(spec["K"]); injected.K.unit=u.keV**-1*u.cm**-2*u.s**-1
    injected.piv.value=float(spec["pivot_keV"]); injected.piv.unit=u.keV
    injected.xc.value=float(spec["cutoff_keV"]); injected.xc.unit=u.keV
    injected.index.value=float(spec["index"])
    pivot=float(case.get("fit_pivot_keV",200.0))
    fit_shape=Cutoff_powerlaw()
    fit_shape.K.value=float(spec["K"])*(pivot/float(spec["pivot_keV"]))**float(spec["index"]); fit_shape.K.unit=injected.K.unit
    fit_shape.piv.value=pivot; fit_shape.piv.unit=u.keV
    fit_shape.xc.value=float(spec["cutoff_keV"]); fit_shape.xc.unit=u.keV
    fit_shape.index.value=float(spec["index"])
    fit_shape.K.min_value=float(spec.get("K_min",1e-10)); fit_shape.K.max_value=float(spec.get("K_max",1.0))
    fit_shape.xc.min_value=float(spec.get("cutoff_min_keV",100.0)); fit_shape.xc.max_value=float(spec.get("cutoff_max_keV",10000.0))
    fit_shape.index.min_value=float(spec.get("index_min",-5.0)); fit_shape.index.max_value=float(spec.get("index_max",1.0))
    fit_shape.index.fix=not bool(spec.get("fit_index_free",False)); fit_shape.xc.fix=not bool(spec.get("fit_cutoff_free",True))
    source_name=str(case.get("source_name","NGC1068"))
    model=Model(PointSource(source_name,l=float(case["longitude_deg"]),b=float(case["latitude_deg"]),spectral_shape=fit_shape))
    full_orientation=open_spacecraft_history(manifest["orientation_file_3m"])
    coord=SkyCoord(l=float(case["longitude_deg"]),b=float(case["latitude_deg"]),frame="galactic",unit="deg")
    gti=GoodTimeInterval.from_pointing_cut(coord,full_orientation,float(case.get("fov_cut_deg",60))*u.deg,earth_occ=False)
    orientation=scale_spacecraft_livetime(full_orientation.apply_gti(gti),exposure/3.0)
    plugin=COSIPlugin(dataset_name,dr=manifest["response_file"],data=data_hist.project("Em","Phi","PsiChi"),bkg=background.project("Em","Phi","PsiChi"),sc_orientation=orientation,nuisance_param=make_cosi_background_parameter(dataset_name),background_pseudocount=BACKGROUND_PSEUDOCOUNT,earth_occ=True)
    plugin.set_model(model)
    like=JointLikelihood(model,DataList(plugin),verbose=False); like.fit()
    source_ts,_=cosi_source_detection_ts(like,plugin)
    results=like.results; shape=results.optimized_model[source_name].spectrum.main.shape
    def rv_or_value(parameter):
        return results.get_variates(parameter.path) if parameter.path in results.optimized_model.free_parameters else float(parameter.value)
    fit_pivot=float(shape.piv.value)
    def evaluate(energy,K,xc,index): return K*(energy/fit_pivot)**index*np.exp(-energy/xc)
    propagator=results.propagate(evaluate,K=rv_or_value(shape.K),xc=rv_or_value(shape.xc),index=rv_or_value(shape.index))
    energy=np.geomspace(100.0,10000.0,160); median=np.zeros_like(energy); low=np.zeros_like(energy); high=np.zeros_like(energy)
    for i,e in enumerate(energy):
        d=propagator(float(e)); median[i]=float(d.median); low[i],high[i]=d.equal_tail_interval(cl=0.68)
    injected_flux=np.asarray([injected.evaluate_at(e) for e in energy],float)
    return dict(manifest=manifest,case=case,exposure=exposure,source_expectation=source_expectation,data=data_hist,background=background,injected=injected,like=like,source_ts=source_ts,energy=energy,median=median,low=low,high=high,injected_flux=injected_flux)


In [ ]:
fits={}
for label,path in manifest_paths.items():
    fits[label]=fit_cpl_manifest(path,"ngc1068_"+label)
    print(label,"source TS =",fits[label]["source_ts"])
    display(fits[label]["like"].results.get_data_frame())


In [ ]:
save_agn_fit_summary(
    output_path=FIT_SUMMARY_PATH,
    fit_results={k:v["like"].results for k,v in fits.items()},
    injected_models={k:{"spectrum":v["injected"]} for k,v in fits.items()},
    ts_values={k:v["source_ts"] for k,v in fits.items()},
    exposure_months={k:v["exposure"] for k,v in fits.items()},
    notes="Explicit single-CPL fits; no shared compact NGC 1068 fitting helper.",
)
print("Saved:",FIT_SUMMARY_PATH)


# SED


In [ ]:
sed_products={}
for label in ("low_flux_24m","testagrossa_24m"):
    fit=fits[label]
    representative=fit_representative_sed(source_expectation=fit["source_expectation"],background_expectation=fit["background"],data_histogram=fit["data"],injected_shape=fit["injected"],n_sed_bins=N_SED_BINS,energy_min_keV=100,energy_max_keV=10000,background_pseudocount=BACKGROUND_PSEUDOCOUNT,detection_ts=4.0)
    representative=classify_representative_sed(representative,detection_ts=4.0)
    _,summary_df,_,summary_file=run_or_load_manifest_sed_ensemble(manifest=fit["manifest"],source_expectation=fit["source_expectation"],background_expectation=fit["background"],injected_shape=fit["injected"],n_sed_bins=N_SED_BINS,energy_min_keV=100,energy_max_keV=10000,background_pseudocount=BACKGROUND_PSEUDOCOUNT,recompute=SED_ENSEMBLE_RECOMPUTE,detection_ts=4.0,expected_seed_count=300)
    sed_products[label]={"representative":representative,"ensemble":ensemble_summary_to_sed_dataframe(summary_df)}


In [ ]:
FONT_SIZE = 25
PAIR_FIGURE_SIZE = (24, 9)

plt.rcParams["agg.path.chunksize"] = 10000
plt.rcParams.update({
    "font.size": FONT_SIZE,
    "font.family": "Times New Roman",
    "font.weight": "550",
    "axes.linewidth": 1.5,
    "axes.labelsize": FONT_SIZE,
    "axes.labelweight": "550",
    "axes.titleweight": "550",
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
})

def style_pair_axis(axis, *, show_ylabel):
    axis.tick_params(
        axis="x", which="major", size=12, width=1.5, direction="in",
        top=True, bottom=True, pad=8, labelsize=FONT_SIZE,
    )
    axis.tick_params(
        axis="x", which="minor", size=6, width=1.5, direction="in",
        top=True, bottom=True,
    )
    axis.tick_params(
        axis="y", which="major", size=12, width=1.5, direction="in",
        left=True, right=True, pad=8, labelsize=FONT_SIZE,
    )
    axis.tick_params(
        axis="y", which="minor", size=6, width=1.5, direction="in",
        left=True, right=True,
    )
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.5)
    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set_xlim(0.085, 12.0)
    axis.set_xlabel("Energy (MeV)", fontsize=FONT_SIZE, fontweight="550")
    axis.set_ylabel(
        r"Energy Flux (erg cm$^{-2}$ s$^{-1}$)" if show_ylabel else "",
        fontsize=FONT_SIZE, fontweight="550",
    )

def plot_pair(kind, save_path=None):
    labels = [
        ("low_flux_24m", "Thermal CPL, K/100", "#D55E00"),
        ("testagrossa_24m", "Testagrossa-reference CPL, K/1000", "#2A8BC3"),
    ]
    fig, axes = plt.subplots(
        1, 2, figsize=PAIR_FIGURE_SIZE, sharex=True, sharey=True,
        constrained_layout=True,
    )
    for panel_index, (axis, (label, title, color)) in enumerate(zip(axes, labels)):
        fit = fits[label]
        frame = sed_products[label][kind]
        roles = frame["plot_role"].astype(str)
        detections = frame.loc[roles == "detection"]
        upper_limits = frame.loc[roles == "upper_limit"]
        energy_values = fit["energy"]

        style_pair_axis(axis, show_ylabel=(panel_index == 0))
        axis.plot(
            energy_values*KEV_TO_MEV,
            SED_KEV_TO_ERG*energy_values**2*fit["injected_flux"],
            color=color, ls=":", lw=3,
        )
        axis.plot(
            energy_values*KEV_TO_MEV,
            SED_KEV_TO_ERG*energy_values**2*fit["median"],
            color=color, lw=2,
        )
        axis.fill_between(
            energy_values*KEV_TO_MEV,
            SED_KEV_TO_ERG*energy_values**2*fit["low"],
            SED_KEV_TO_ERG*energy_values**2*fit["high"],
            color=color, alpha=0.18,
        )

        def xerr(frame_part):
            return KEV_TO_MEV*np.vstack([
                frame_part["e_ref_keV"]-frame_part["e_min_keV"],
                frame_part["e_max_keV"]-frame_part["e_ref_keV"],
            ])

        if not detections.empty:
            y = detections["sed_erg_cm2_s"].to_numpy(float)
            axis.errorbar(
                detections["e_ref_keV"]*KEV_TO_MEV, y, xerr=xerr(detections),
                yerr=np.vstack([
                    np.maximum(y-detections["sed_lo_erg_cm2_s"], 0.0),
                    np.maximum(detections["sed_hi_erg_cm2_s"]-y, 0.0),
                ]),
                fmt="o", color=color, ecolor=color, markerfacecolor="white",
                markeredgecolor=color, markeredgewidth=1.6, elinewidth=1.5,
                capsize=4, markersize=8, zorder=5,
            )
        if not upper_limits.empty:
            if "sed_ul95_erg_cm2_s" in upper_limits:
                y = upper_limits["sed_ul95_erg_cm2_s"].to_numpy(float)
            elif "sed_ul95_median_erg_cm2_s" in upper_limits:
                y = upper_limits["sed_ul95_median_erg_cm2_s"].to_numpy(float)
            else:
                y = upper_limits["sed_hi_erg_cm2_s"].to_numpy(float)
            axis.errorbar(
                upper_limits["e_ref_keV"]*KEV_TO_MEV, y, xerr=xerr(upper_limits),
                yerr=np.maximum(0.5*y, np.finfo(float).tiny), uplims=True,
                fmt="v", color=color, ecolor=color, markerfacecolor="white",
                markeredgecolor=color, markeredgewidth=1.8, elinewidth=2.4,
                capsize=6, markersize=11, barsabove=True, zorder=6,
            )
        axis.text(
            0.98, 0.97, "NGC 1068\n"+title+"\n24-month",
            transform=axis.transAxes, ha="right", va="top",
            fontsize=FONT_SIZE, fontweight="550",
        )

    axes[0].legend(handles=[
        plt.Line2D([], [], color="#D55E00", ls=":", lw=3, label="Injected"),
        plt.Line2D([], [], color="#D55E00", lw=2, label="Best fit & 68% band"),
        plt.Line2D([], [], color="0.25", marker="o", markerfacecolor="white",
                   linestyle="None", markersize=8, label="COSI SED"),
        plt.Line2D([], [], color="0.25", marker="v", markerfacecolor="white",
                   linestyle="None", markersize=10, label="95% upper limit"),
    ], fontsize=FONT_SIZE, loc="lower left", frameon=False)
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
        print("Saved:", save_path)
    return fig, axes


### Plot A — representative median data sets


In [ ]:
fig_representative,axes_representative=plot_pair("representative",PLOT_DIR/"NGC1068_Case1_lowFlux100_vs_Testagrossa_SED_RepresentativeMedianDataset.pdf")
plt.show()


### Plot B — 300-seed median SEDs


In [ ]:
fig_ensemble,axes_ensemble=plot_pair("ensemble", None)  # Plot B is displayed but intentionally not saved.
plt.show()
